# Serving layer: the decode-speed prediction against recorded use

Before a generation runs, the forked Ollama predicts its decode speed and sends it on that generation's `gen.end`
frame as `predicted_decode.ms_per_token`. `basis` is `profile` (the card's measured bandwidth and per-token costs) or
`profile_corrected` (that, scaled by the model's own earlier clean generations: `correction_samples` of them). The same
frame carries what happened: `timings.eval_ms` over `timings.decoded` tokens. This notebook compares the two for every
generation the bench store holds (`docs/FORKED-BACKENDS.md`, "Measured decode speed").

In [1]:
import json
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "bench"
DATA = HERE / "data" / "serving-decode"
pd.set_option("display.width", 160)

_con = None
def con():
    """The store, opened only when an extract has to read it (a pinned run never connects)."""
    global _con
    _con = _con or wmlnb.store()
    return _con

## The rows, pinned

Every `gen.end` frame that carries a prediction, one per frame: the `box_frames` view keeps one row per
(box, server_at, kind, digest), since overlapping pushes put a frame in the store more than once.

In [2]:
GEN_END = '''
SELECT box, "at", frame FROM box_frames
WHERE kind = 'gen.end' AND frame LIKE '%predicted_decode%' ORDER BY "at"'''

def gen_ends():
    out = []
    for box, at, frame in con().sql(GEN_END).fetchall():
        f = json.loads(frame)
        p, t = f.get("predicted_decode") or {}, f.get("timings") or {}
        out.append({"box": box, "at": at, "model": f.get("model"), "basis": p.get("basis"), "predicted_ms": p.get("ms_per_token"),
                    "profile_ms": p.get("profile_ms_per_token"), "correction": p.get("correction_factor"),
                    "samples": p.get("correction_samples"), "decoded": t.get("decoded"), "eval_ms": t.get("eval_ms")})
    return out

gens = pd.DataFrame(wmlnb.extract(DATA / "gen-end.json", gen_ends,
                                  source={"store": "bench store (sync.mjs)", "view": "box_frames", "query": GEN_END.strip()}))
pd.DataFrame(wmlnb.provenance(DATA / "gen-end.json"))

,extract,sha256,rows,extracted_at,commit,query,store,view
0,notebooks/bench/data/serving-decode/gen-end.json,9b4d658f2752968f,10378,2026-10-21T13:42:17+00:00,7d81e7ccfdb0,"SELECT box, ""at"", frame FROM box_frames\nWHERE...",bench store (sync.mjs),box_frames


## The error

Actual speed is `eval_ms / decoded`. A generation of fewer than 32 tokens is left out: its decode time is a handful of
milliseconds, and a scheduling hiccup dominates it. The error is `(predicted - actual) / actual`, so a negative error is
a prediction that was too fast.

In [3]:
g = gens.dropna(subset=["predicted_ms", "decoded", "eval_ms"])
g = g[(g.decoded >= 32) & (g.eval_ms > 0) & (g.predicted_ms > 0)].copy()
g["actual_ms"] = g.eval_ms / g.decoded
g["err"] = (g.predicted_ms - g.actual_ms) / g.actual_ms
g["abs_err"] = g.err.abs()
first, last = pd.to_datetime(g["at"].min(), unit="ms"), pd.to_datetime(g["at"].max(), unit="ms")
print(f"{len(gens)} gen.end frames with a prediction; {len(g)} with >= 32 decoded tokens, {g.model.nunique()} models, {first:%Y-%m-%d} to {last:%Y-%m-%d}")

def summary(d):
    return pd.Series({"n": len(d), "median |err| %": round(d.abs_err.median() * 100, 1), "median signed %": round(d.err.median() * 100, 1),
                      "within 10 %": round((d.abs_err <= 0.10).mean() * 100), "p90 |err| %": round(d.abs_err.quantile(0.9) * 100, 1)})

pd.concat({"all": summary(g), **{b: summary(d) for b, d in g.groupby("basis")}}, axis=1).T

10378 gen.end frames with a prediction; 8613 with >= 32 decoded tokens, 13 models, 2026-10-09 to 2026-10-21


,n,median |err| %,median signed %,within 10 %,p90 |err| %
all,8613.0,1.4,-0.6,75.0,43.1
profile,165.0,74.3,-74.3,6.0,79.9
profile_corrected,8448.0,1.3,-0.5,77.0,41.8


The corrected prediction is the one the server makes once a model has clean generations to learn from, and almost
every generation has it. The uncorrected `profile` basis runs too fast by about three quarters: the card's raw
bandwidth model alone is not a usable prediction, and the correction is what makes it one.

## By model

Where the long tail comes from.

In [4]:
(g.groupby("model").apply(summary, include_groups=False)
 .astype({"n": int}).sort_values("n", ascending=False))

,n,median |err| %,median signed %,within 10 %,p90 |err| %
model,,,,,
registry.ollama.ai/library/glm-4.7-flash:latest,3432,1.0,-0.3,84.0,39.6
registry.ollama.ai/library/qwen3.6:35b,2286,1.8,-0.6,79.0,41.7
registry.ollama.ai/library/qwen3-vl:30b,2054,1.1,-0.6,71.0,57.3
registry.ollama.ai/library/gemma4:31b,541,14.3,-13.9,38.0,36.1
registry.ollama.ai/library/qwen3.8-flash-next:vision,115,6.7,-4.9,63.0,34.9
registry.ollama.ai/library/qwen3.5:4b,36,65.5,-65.5,14.0,78.0
registry.ollama.ai/library/qwen3.8:27b,28,29.6,-26.9,7.0,48.7
registry.ollama.ai/library/nemotron-3-nano:4b,27,74.5,-74.5,44.0,74.7
registry.ollama.ai/library/qwen3.5:9b,27,77.9,-77.9,11.0,79.2
